In [ ]:
%pip install pandas numpy matplotlib scikit-learn joblib

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [ ]:
customers = pd.read_csv("../data/processed/customers_features.csv")

print(customers.shape)
print(customers.columns.tolist())

In [ ]:
# import pandas as pd
# customers_features = pd.read_csv("data/processed/customers_features.csv")
import pandas as pd

customers_features = pd.read_csv("../data/processed/customers_features.csv")
customer_df = customers_features.copy()
print(customer_df.shape)


print(customer_df.columns.tolist())

customer_df[
    [
        "customer_total_orders",
        "customer_total_spent",
        "customer_avg_order_value",
        "repeat_customer"
    ]
].describe()

In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

print("Orders:")
orders_features = pd.read_csv(PROCESSED_DIR / "orders_features.csv")
print(orders_features[["order_id", "customer_id"]].head())

print("\nOrder Items:")
order_items_features = pd.read_csv(PROCESSED_DIR / "order_items_features.csv")
print(order_items_features[["order_id", "price"]].head())

print("\nOrders shape:", orders_features.shape)
print("Order items shape:", order_items_features.shape)

In [ ]:
print("Unique customers:",
      orders_features["customer_id"].nunique())

print("Unique orders:",
      orders_features["order_id"].nunique())

print("Duplicate order IDs:",
      orders_features["order_id"].duplicated().sum())

In [ ]:
customer_order_counts = (
    orders_features
    .groupby("customer_id")["order_id"]
    .nunique()
)

print(customer_order_counts.describe())

print("\nCustomers with >1 order:")
print((customer_order_counts > 1).sum())

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

from pathlib import Path

# Project root
BASE_DIR = Path.cwd().parent

# Database path
DATABASE = BASE_DIR / "database" / "retail.db"

print("Database:", DATABASE)

conn = sqlite3.connect(DATABASE)

customers = pd.read_sql("SELECT * FROM customers", conn)
orders = pd.read_sql("SELECT * FROM orders", conn)
order_items = pd.read_sql("SELECT * FROM order_items", conn)

conn.close()

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)

In [ ]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nORDER ITEMS")
print(order_items.columns.tolist())

In [ ]:
customer_orders = orders.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left"
)

print(customer_orders.shape)

In [ ]:
print(
    "Unique customers:",
    customer_orders["customer_unique_id"].nunique()
)

In [ ]:
order_revenue = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_revenue=("price", "sum")
    )
)

print(order_revenue.head())

In [ ]:
customer_orders = customer_orders.merge(
    order_revenue,
    on="order_id",
    how="left"
)

customer_orders["order_revenue"] = (
    customer_orders["order_revenue"]
    .fillna(0)
)

In [ ]:
customer_orders["order_purchase_timestamp"] = pd.to_datetime(
    customer_orders["order_purchase_timestamp"],
    errors="coerce"
)

print(
    customer_orders["order_purchase_timestamp"].min(),
    customer_orders["order_purchase_timestamp"].max()
)

In [ ]:
customer_features = (
    customer_orders
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_spent=("order_revenue", "sum"),
        avg_order_value=("order_revenue", "mean"),
        last_purchase_date=("order_purchase_timestamp", "max"),
        first_purchase_date=("order_purchase_timestamp", "min")
    )
    .reset_index()
)

In [ ]:
customer_features["repeat_customer"] = (
    customer_features["total_orders"] > 1
).astype(int)

In [ ]:
reference_date = customer_features["last_purchase_date"].max()

customer_features["recency_days"] = (
    reference_date -
    customer_features["last_purchase_date"]
).dt.days

In [ ]:
customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase_date"] -
    customer_features["first_purchase_date"]
).dt.days

In [ ]:
print(customer_features.shape)

print(
    customer_features[
        [
            "total_orders",
            "total_spent",
            "avg_order_value",
            "repeat_customer",
            "recency_days"
        ]
    ].describe()
)

In [ ]:
print(
    customer_features["repeat_customer"].value_counts()
)

In [ ]:
print(
    customer_features[
        customer_features["total_orders"] > 1
    ].head(10)
)

In [ ]:
segmentation_features = [
    "total_orders",
    "total_spent",
    "avg_order_value",
    "recency_days"
]

In [ ]:
X = customer_features[segmentation_features].copy()

print(X.shape)
print(X.head())

In [ ]:
print(X.isnull().sum())

In [ ]:
X = X.fillna(0)

In [ ]:
print(X.isnull().sum())

In [ ]:
X_log = X.copy()

X_log["total_orders"] = np.log1p(
    X_log["total_orders"]
)

X_log["total_spent"] = np.log1p(
    X_log["total_spent"]
)

X_log["avg_order_value"] = np.log1p(
    X_log["avg_order_value"]
)

X_log["recency_days"] = np.log1p(
    X_log["recency_days"]
)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_log)

print(X_scaled.shape)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [ ]:
inertia = []

K_range = range(2, 9)

for k in K_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    inertia.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    K_range,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Customer Segmentation")

plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

customer_features["cluster"] = kmeans.fit_predict(
    X_scaled
)

In [ ]:
print(
    customer_features["cluster"].value_counts()
)

In [ ]:
cluster_summary = (
    customer_features
    .groupby("cluster")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_orders=("total_orders", "mean"),
        avg_spending=("total_spent", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        avg_recency=("recency_days", "mean"),
        repeat_rate=("repeat_customer", "mean")
    )
    .round(2)
)

cluster_summary

In [ ]:
cluster_names = {
    0: "Low-Value / Inactive",
    1: "New / Recent",
    2: "VIP / Loyal",
    3: "At-Risk / High-Value"
}

customer_features["customer_segment"] = (
    customer_features["cluster"].map(cluster_names)
)

In [ ]:
print(
    customer_features[
        [
            "customer_unique_id",
            "total_orders",
            "total_spent",
            "avg_order_value",
            "recency_days",
            "repeat_customer",
            "cluster",
            "customer_segment"
        ]
    ].head(20)
)

In [ ]:
print(
    customer_features["customer_segment"].value_counts()
)

In [ ]:
segment_summary = (
    customer_features
    .groupby("customer_segment")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_orders=("total_orders", "mean"),
        avg_spending=("total_spent", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        avg_recency=("recency_days", "mean"),
        repeat_rate=("repeat_customer", "mean")
    )
    .round(2)
    .sort_values("avg_spending", ascending=False)
)

segment_summary

In [ ]:
segment_summary["repeat_rate"] = (
    segment_summary["repeat_rate"] * 100
).round(2)

segment_summary

In [ ]:
plt.figure(figsize=(10, 6))

for segment in customer_features["customer_segment"].unique():

    subset = customer_features[
        customer_features["customer_segment"] == segment
    ]

    plt.scatter(
        subset["total_orders"],
        subset["total_spent"],
        alpha=0.4,
        label=segment
    )

plt.xlabel("Total Orders")
plt.ylabel("Total Spending")
plt.title("Customer Segmentation using K-Means")
plt.legend()

plt.show()

In [ ]:
OUTPUT_DIR = BASE_DIR / "data" / "processed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_file = OUTPUT_DIR / "customer_segments.csv"

customer_features.to_csv(
    output_file,
    index=False
)

print(f"Saved successfully: {output_file}")

In [ ]:
customer_df.head()

In [ ]:
customer_df.columns.tolist()

In [ ]:
print("Final Customer Segment Distribution")
print("=" * 50)

segment_distribution = (
    customer_features["customer_segment"]
    .value_counts()
    .rename_axis("customer_segment")
    .reset_index(name="customers")
)

segment_distribution["percentage"] = (
    segment_distribution["customers"]
    / len(customer_features)
    * 100
).round(2)

segment_distribution

In [ ]:
final_segment_summary = (
    customer_features
    .groupby("customer_segment")
    .agg(
        customers=("customer_unique_id", "nunique"),
        avg_orders=("total_orders", "mean"),
        avg_spending=("total_spent", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        avg_recency=("recency_days", "mean"),
        repeat_rate=("repeat_customer", "mean")
    )
    .reset_index()
)

final_segment_summary["customer_percentage"] = (
    final_segment_summary["customers"]
    / final_segment_summary["customers"].sum()
    * 100
).round(2)

final_segment_summary["repeat_rate"] = (
    final_segment_summary["repeat_rate"] * 100
).round(2)

final_segment_summary = final_segment_summary.round(2)

final_segment_summary

In [ ]:
summary_file = OUTPUT_DIR / "customer_segment_summary.csv"

final_segment_summary.to_csv(
    summary_file,
    index=False
)

print(f"Segment summary saved to: {summary_file}")

In [ ]:
import joblib

In [ ]:
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    kmeans,
    MODEL_DIR / "customer_kmeans.pkl"
)

joblib.dump(
    scaler,
    MODEL_DIR / "customer_segmentation_scaler.pkl"
)

print("K-Means model and scaler saved successfully.")

In [ ]:
segment_interpretation = pd.DataFrame({
    "customer_segment": [
        "Low-Value / Inactive",
        "New / Recent",
        "VIP / Loyal",
        "At-Risk / High-Value"
    ],
    "business_action": [
        "Re-engagement campaigns and discounts",
        "Onboarding and second-purchase campaigns",
        "VIP rewards, loyalty programs and premium offers",
        "Win-back campaigns and personalized offers"
    ]
})

segment_interpretation

In [ ]:
plt.figure(figsize=(10, 6))

for segment in customer_features["customer_segment"].unique():

    subset = customer_features[
        customer_features["customer_segment"] == segment
    ]

    plt.scatter(
        subset["total_orders"],
        subset["total_spent"],
        alpha=0.4,
        label=segment
    )

plt.xlabel("Total Orders")
plt.ylabel("Total Spending")
plt.title("Customer Segmentation: Orders vs Spending")
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
print("========== CUSTOMER SEGMENTATION FINAL CHECK ==========")

print("Total customers:", len(customer_features))

print(
    "Unique customers:",
    customer_features["customer_unique_id"].nunique()
)

print(
    "Number of clusters:",
    customer_features["cluster"].nunique()
)

print("\nCluster counts:")
print(
    customer_features["cluster"]
    .value_counts()
    .sort_index()
)

print("\nBusiness segments:")
print(
    customer_features["customer_segment"]
    .value_counts()
)

print("\nMissing values:")
print(
    customer_features[
        [
            "total_orders",
            "total_spent",
            "avg_order_value",
            "recency_days",
            "repeat_customer",
            "cluster",
            "customer_segment"
        ]
    ].isnull().sum()
)

In [ ]:
customer_features["cluster"].value_counts()

In [ ]:
customer_features.columns.tolist()

In [ ]:
import os

# Create ML output folder if it doesn't exist
os.makedirs("../outputs/ml", exist_ok=True)

# Select final customer segmentation columns
customer_segments = customer_features[
    [
        "customer_unique_id",
        "total_orders",
        "total_spent",
        "avg_order_value",
        "last_purchase_date",
        "first_purchase_date",
        "repeat_customer",
        "recency_days",
        "customer_lifetime_days",
        "cluster",
        "customer_segment"
    ]
].copy()

# Save
customer_segments.to_csv(
    "../outputs/ml/customer_segments.csv",
    index=False
)

print("Customer segmentation file saved successfully.")
print("Shape:", customer_segments.shape)
print(customer_segments.head())

In [ ]:
customer_segments["cluster"].value_counts().sort_index()

In [ ]:
customer_segments["customer_segment"].value_counts()

In [ ]:
import os

print(os.path.exists("../outputs/ml/customer_segments.csv"))

In [ ]:
pd.read_csv("../outputs/ml/customer_segments.csv").shape

In [ ]:
customer_segments["customer_segment"].value_counts()

In [ ]:
from pathlib import Path
import pandas as pd

# 1. Automatically find all CSV files in the project (excluding .venv)
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
csv_files = [
    f for f in project_dir.rglob("*.csv")
    if ".venv" not in f.parts and f.stat().st_size > 0
]

if not csv_files:
    raise FileNotFoundError(
        f"No CSV dataset files found inside '{project_dir}'. "
        "Please drag and drop your dataset CSV file into your project folder."
    )

# 2. Pick the largest CSV file (your main dataset)
data_path = max(csv_files, key=lambda f: f.stat().st_size)
print(f"Loaded Dataset: {data_path.resolve()}")

customer_segments = pd.read_csv(data_path)

# 3. Identify Segment / Cluster Column
possible_segment_cols = [
    "customer_segment", "segment", "Segment", 
    "cluster", "Cluster", "cluster_label", "KMeans_Cluster"
]
segment_col = next((c for c in possible_segment_cols if c in customer_segments.columns), None)

# If no segment column exists, show available columns and preview
if segment_col is None:
    print("\n⚠️ No segment/cluster column found in this file.")
    print("Columns available in your CSV:", list(customer_segments.columns))
    print("\nPreview of loaded data:")
    display(customer_segments.head())
else:
    print(f"\nGrouping data by: '{segment_col}'")
    
    # Build dynamic aggregation dictionary based on available columns
    agg_dict = {}
    id_col = next((c for c in ["customer_unique_id", "customer_id", "id"] if c in customer_segments.columns), None)
    if id_col:
        agg_dict["customers"] = (id_col, "nunique")

    for col_name, metric_name in [
        ("total_orders", "avg_orders"),
        ("total_spent", "avg_spending"),
        ("avg_order_value", "avg_order_value"),
        ("recency_days", "avg_recency"),
        ("repeat_customer", "repeat_rate")
    ]:
        if col_name in customer_segments.columns:
            agg_dict[metric_name] = (col_name, "mean")

    if agg_dict:
        segment_summary = (
            customer_segments.groupby(segment_col)
            .agg(**agg_dict)
            .round(2)
            .reset_index()
        )
    else:
        segment_summary = (
            customer_segments.groupby(segment_col)
            .mean(numeric_only=True)
            .round(2)
            .reset_index()
        )

    display(segment_summary)

In [ ]:
customer_segments["customer_segment"].value_counts()

In [ ]:
print(customer_features.columns.tolist())

In [ ]:
customer_features["customer_segment"].value_counts()

In [ ]:
segment_summary = customer_features.groupby("customer_segment").agg(
    customers=("customer_unique_id", "count"),
    avg_orders=("total_orders", "mean"),
    avg_spending=("total_spent", "mean"),
    avg_order_value=("avg_order_value", "mean"),
    avg_recency=("recency_days", "mean"),
    repeat_rate=("repeat_customer", "mean")
).round(2)

segment_summary

In [ ]:
print(customer_features.columns.tolist())

In [ ]:
customer_features["cluster"].value_counts()

In [ ]:
customer_features["customer_segment"].value_counts()

In [ ]:
customer_features.to_csv(
    "../outputs/ml/customer_segments.csv",
    index=False
)

print("Customer segmentation saved successfully.")
print("Shape:", customer_features.shape)